In [1]:
!pip install mlflow boto3 awscli optuna imbalanced-learn lightgbm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.

In [2]:
# AWS Access Key
!aws configure

AWS Access Key ID [None]: 
^C


In [3]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-52-204-122-132.compute-1.amazonaws.com:5000/")

In [ ]:
# Set or create an experiment
mlflow.set_experiment("LightGBM HP Tuning")

In [ ]:
import pandas as pd
df= pd.read_csv("reddit_preproccessing.csv").dropna()
df.shape

In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt

In [ ]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category']=df['category'].map({-1: 2, 0:0, 1:1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

In [ ]:
# Step 3: TF-IDF vectorizer setup
ngram_range= (1, 3)
max_features= 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df["clean_comment"])
y= df["category"]

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [5]:
# Step 5: Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2,random_state=42)

NameError: name 'X_resampled' is not defined

In [ ]:
# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test,
               params, trial_number):
    with mlflow.start_run():
        # Tags
        mlflow.set_tag(
            "mlflow.runName",
            f"{model_name}_trial_{trial_number}"
        )
        mlflow.set_tag(
            "experiment_type",
            "lightgbm_hyperparameter_tuning"
        )
        # Log algorithm name
        mlflow.log_param("algo_name", model_name)
        # Log Optuna hyperparameters
        mlflow.log_params(params)

        # Train model
        model.fit(X_train, y_train)
        # Predictions
        y_pred = model.predict(X_test)

        # Accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Classification report
        classification_rep = classification_report(
            y_test,
            y_pred,
            output_dict=True
        )
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(
                        f"{label}_{metric}",
                        value
                    )

        # Log model
        mlflow.sklearn.log_model(
            model,
            f"{model_name}_model"
        )

        return accuracy

In [ ]:
# Step 6: Optuna objective function for LightGBM
def objective_lightgbm(trial):
  # Hyperparameter space to explore
  n_estimators = trial.suggest_int("n_estimators", 100, 1000)
  learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
  max_depth = trial.suggest_int("max_depth", 3, 15)
  num_leaves = trial.suggest_int("num_leaves", 20, 150)
  min_child_samples = trial.suggest_int("min_child_samples", 10, 100)
  colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
  subsample = trial.suggest_float("subsample", 0.5, 1.0)
  reg_alpha = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True) # L1 regularization
  reg_lambda = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True) # L2 regularization

  # Log trial parameters
  params={
      'n_estimators' : n_estimators,
      'learning_rate' : learning_rate,
      "max_depth" : max_depth,
      "num_leaves" : num_leaves,
      "min_child_samples" : min_child_samples,
      "colsample_bytree" : colsample_bytree,
      "subsample" : subsample,
      "reg_alpha" : reg_alpha,
      "reg_lambda" : reg_lambda
  }

  # Create LightGBM model
  model = LGBMClassifier(n_estimators=n_estimators,
                         learning_rate=learning_rate,
                         max_depth=max_depth,
                         num_leaves=num_leaves,
                         min_child_samples=min_child_samples,
                         colsample_bytree=colsample_bytree,
                         subsample=subsample,
                         reg_alpha=reg_alpha,
                         reg_lambda=reg_lambda,
                         random_state=42)

  # Log each trial as as separate run in MLflow
  accuracy = log_mlflow("LightGBM", model, X_train, X_test, y_train, y_test, params, trial.number)

  return accuracy

In [11]:
# Step 7: Run Optuna for LightGBM, log the best model, and plot the importance of each parameter
def run_optuna_experiment():
  study= optuna.create_study(direction="maximize")
  study.optimize(objective_lightgbm, n_trials=100) # Inceased to 100 trials

  # Get the best parameters
  best_params= study.best_params
  best_model=LGBMClassifier(n_estimators=best_params["n_estimators"],
                            learing_rate=best_params["learing_rate"],
                            max_depth=best_params["max_depth"],
                            num_leaves=best_params["num_leaves"],
                            min_child_samples=best_params["min_child_samples"],
                            colsample_bytree=best_params["colsample_bytree"],
                            subsample=best_params["subsample"],
                            reg_alpha=best_params["reg_alpha"],
                            reg_lambda=best_params["reg_lambda"],
                            random_state=42
                            )

  # Log the best model with MLflow and print the classification report
  log_mlflow("LighGBM", best_model, X_train , X_test, y_train, y_test, best_params, "Best")

  # Plot parameter importance
  optuna.visualization.plot_param_importances(study).show()

  # Plot optimization history
  optuna.visualization.plot_optimization_history(study).show()

In [12]:
# Run optimization for LightGBM
run_optuna_experiment()

[I 2026-06-05 12:55:32,298] A new study created in memory with name: no-name-67455215-0105-4d2a-a32c-108d93aec1f0


NameError: name 'objective_lightgbm' is not defined